# Clase 17 asíncrona: listas ligadas, BFS y 0-1 BFS

## Pregunta inicial

**¿Qué estructura necesitamos cuando todos los pendientes tienen la misma prioridad, y qué cambia cuando existen dos niveles de prioridad?**

<div style="border:1px solid #9bb8d3;border-left:6px solid #315f8c;background:#eef5fb;padding:14px 18px;margin:14px 0;border-radius:4px"><strong>INSTRUCCIÓN — Cuaderno de trabajo</strong><br>Predice antes de avanzar, dibuja los enlaces y registra cada respuesta en <code>notebook.md</code>. No respondas dentro de este archivo.</div>

**Responde esta pregunta en notebook.md.**


## Objetivos de aprendizaje

Al terminar esta ruta podrás:

- distinguir nodo, lista ligada y estructura contenedora;
- implementar una cola ligada simple con frente, final y tamaño consistentes;
- explicar por qué BFS usa FIFO y marcar visitados al encolar;
- reconstruir caminos desde predecesores con contratos robustos;
- implementar una deque con lista doble y operar en ambos extremos en O(1);
- aplicar 0-1 BFS con tu propia deque y justificar cada extremo de inserción;
- normalizar grafos sin mutarlos y clasificar errores de tipo, valor y clave;
- comparar BFS, 0-1 BFS y Dijkstra desde su operación dominante;
- diseñar pruebas que protejan invariantes y casos frontera.

### Ruta sugerida para cuatro horas

1. **Comprender y predecir** — secciones 1 a 8.
2. **Construir BFS** — secciones 9 a 13.
3. **Construir deque y 0-1 BFS** — secciones 14 a 20.
4. **Implementar, probar y sintetizar** — secciones 21 a 25.


## 1. Presentación de la clase

Imagina un centro de soporte. Cada solicitud pendiente tiene el mismo nivel de urgencia y debe atenderse en el orden en que llegó. Un poco después aparece otra regla: algunas acciones cuestan **cero** —pueden resolverse de inmediato— y otras cuestan **uno**. La pregunta ya no es solo «¿qué llegó primero?», sino «¿qué candidato conviene procesar antes?».

Esta clase sigue una cadena deliberada: primero construiremos una cola con nodos; después la usaremos dentro de BFS. Luego agregaremos enlaces hacia atrás, construiremos una deque y la usaremos en 0-1 BFS. El propósito no es implementar listas generales: solo las operaciones que los algoritmos realmente necesitan.

```text
misma prioridad       dos prioridades         prioridad general
cola → BFS            deque → 0-1 BFS         heap → Dijkstra
```

<div style="border-left:5px solid #315f8c;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong>RUTA — Cuatro horas asíncronas</strong><br>Lee la explicación, predice cada visualización, traza a mano, implementa los contratos y termina con pruebas y discusión. Registra tus respuestas en <code>notebook.md</code>.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué aspecto del problema cambia cuando pasamos de una sola prioridad a dos prioridades?

**Responde esta pregunta en notebook.md.**


## 2. Problema inicial con pop(0)

Una primera versión podría guardar pendientes en una lista de Python:

```python
pendientes = ["A", "B", "C"]
actual = pendientes.pop(0)
```

El resultado funcional es correcto: sale `A`. El problema aparece al repetirlo. Los elementos posteriores deben desplazarse una posición; retirar el primero cuesta O(n). Si un recorrido procesa miles de vértices, ese trabajo accidental se repite una y otra vez.

En una cola ligada, en cambio, el frente es una referencia. Desencolar significa mover esa referencia al siguiente nodo; no hay arreglo que compactar. La estructura debe conservar también el final para encolar sin recorrer toda la cadena.

| Representación | Encolar al final | Desencolar al frente |
| --- | ---: | ---: |
| lista con `append` / `pop(0)` | O(1) amortizado | O(n) |
| cola ligada con dos extremos | O(1) | O(1) |

<div style="border-left:5px solid #b26a00;background:#fff6e6;padding:12px 16px;margin:14px 0"><strong>ADVERTENCIA — Mismo resultado, costo distinto</strong><br>Una salida correcta en un ejemplo pequeño no demuestra que se eligió bien la estructura auxiliar.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué trabajo repetido introduce pop(0) y por qué una referencia al frente lo evita?

**Responde esta pregunta en notebook.md.**


## 3. Nodo y referencias

Un **nodo** es una pieza individual: guarda un valor y una o más referencias. Una **lista ligada** es el conjunto organizado de nodos alcanzables desde una referencia inicial. La **estructura contenedora** —por ejemplo, `ColaLigada`— agrega reglas: qué extremo se modifica, qué operación está permitida y qué invariantes deben mantenerse.

```text
variable frente
      │
      ▼
┌─────────────┐     ┌─────────────┐
│ valor: "A"  │ ──▶ │ valor: "B"  │ ──▶ None
│ siguiente ──┘     │ siguiente ──┘
└─────────────┘     └─────────────┘
```

La referencia no contiene una copia del siguiente nodo: identifica el mismo objeto. Por eso cambiar `actual.siguiente` modifica la cadena. También explica el riesgo de dejar referencias residuales en un nodo retirado: aunque ya no sea parte lógica de la lista, todavía podría mantener accesible el resto.

<div style="border-left:5px solid #315f8c;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong>NOTA — Tres niveles</strong><br>No confundas el valor <code>"A"</code>, el nodo que lo guarda y la cola que administra ese nodo.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Cuál es la diferencia entre el valor guardado, un nodo y la estructura que administra los nodos?

**Responde esta pregunta en notebook.md.**


## 4. Lista ligada simple

En una lista simplemente ligada cada nodo conoce únicamente a su sucesor. Desde el inicio podemos caminar hacia adelante, pero no retroceder sin volver a recorrer desde el principio.

```text
inicio → [A | •] → [B | •] → [C | None]
```

Para esta clase la forma simple basta para una cola: encolar modifica el extremo final y desencolar modifica el extremo inicial. No necesitamos insertar en posiciones arbitrarias, ordenar la lista ni buscar valores. Reducir el alcance protege la idea central: las operaciones se eligen por lo que el algoritmo necesita.

Los casos frontera importan más que el dibujo normal. En una lista vacía `inicio` es `None`. Al agregar el primer nodo, inicio y final deben referirse al mismo objeto. Al retirar el último, ambas referencias vuelven a `None`. Si solo una cambia, la estructura queda en un estado contradictorio.

<div style="border-left:5px solid #7b4bb7;background:#f4effa;padding:12px 16px;margin:14px 0"><strong>IMPORTANTE — Alcance</strong><br>No implementaremos una lista general. Implementaremos exactamente los enlaces que hacen constante el trabajo de cola.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación sería costosa si una lista simple solo guardara la referencia al inicio?

**Responde esta pregunta en notebook.md.**


## 5. Cola ligada

Una cola aplica la política **FIFO**: primero en entrar, primero en salir. `ColaLigada` guarda tres datos: `_frente`, `_final` y `_tamano`. Encolar crea un nodo, lo enlaza después del final anterior y mueve `_final`. Desencolar guarda el frente, avanza `_frente`, separa el nodo retirado y devuelve su valor.

```text
frente                                  final
  │                                       │
  ▼                                       ▼
[A | •] ─────────▶ [B | •] ─────────▶ [C | None]
sale por aquí                        entra por aquí
```

`primero()` observa sin retirar. `esta_vacia()` puede responder con `_tamano == 0`; `len(cola)` devuelve el tamaño almacenado. Actualizar el tamaño en cada operación evita recorrer la cadena para contarlos.

<div style="border-left:5px solid #2f7d5b;background:#edf8f2;padding:12px 16px;margin:14px 0"><strong>CONSEJO — Lee por contrato</strong><br>Antes de escribir enlaces, anota precondición, referencias que cambian, valor devuelto y estado final para los casos vacío, uno y varios nodos.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Por qué necesitamos frente y final para que las dos operaciones dominantes sean O(1)?

**Responde esta pregunta en notebook.md.**


## 6. Invariantes de cola

Un invariante es una afirmación que debe ser cierta antes y después de cada operación pública. No describe solo un ejemplo: define todos los estados válidos.

| Estado | Invariantes obligatorios |
| --- | --- |
| vacía | `frente is None`, `final is None`, tamaño 0 |
| un nodo | `frente is final`, `final.siguiente is None`, tamaño 1 |
| varios | frente alcanza exactamente tamaño nodos; final es el último; `final.siguiente is None` |

Al retirar el último nodo no basta con mover el frente: hay que limpiar también el final. Al retirar cualquier nodo conviene asignar `retirado.siguiente = None`; así el objeto separado no retiene accidentalmente la cadena.

Una prueba puede observar resultados públicos y, cuando el objetivo es proteger la representación, inspeccionar cuidadosamente estos atributos internos. La prueba de reutilización es especialmente poderosa: vacía la cola, vuelve a encolar y comprueba que no reaparece una cadena vieja.

<div style="border-left:5px solid #b42318;background:#fff0ef;padding:12px 16px;margin:14px 0"><strong>PRECAUCIÓN — Último elemento</strong><br>Es el caso que transforma dos referencias no nulas en dos referencias nulas. Trátalo explícitamente.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué tres afirmaciones deben ser simultáneamente ciertas después de desencolar el único elemento?

**Responde esta pregunta en notebook.md.**


## 7. Operaciones manuales

Traza esta secuencia antes de programar. Dibuja cada nodo, las referencias externas y el tamaño. Los primeros pasos sirven de modelo; completa el resto en `notebook.md`.

| Paso | Operación | frente | final | cadena | tamaño | valor devuelto |
| ---: | --- | --- | --- | --- | ---: | --- |
| 0 | crear cola | `None` | `None` | vacía | 0 | — |
| 1 | `encolar("A")` | A | A | A | 1 | — |
| 2 | `encolar("B")` | A | B | A → B | 2 | — |
| 3 | `encolar("C")` | ? | ? | ? | ? | — |
| 4 | `desencolar()` | ? | ? | ? | ? | ? |
| 5 | vaciar | ? | ? | ? | ? | ? |
| 6 | `encolar("Z")` | ? | ? | ? | ? | — |

Ejecuta después la visualización. Antes de pulsar **Siguiente**, predice qué enlace o extremo cambiará. El modo reto no reemplaza tu traza: la utiliza como hipótesis.

<div style="border-left:5px solid #2f7d5b;background:#edf8f2;padding:12px 16px;margin:14px 0"><strong>ACTIVIDAD — Seis flujos</strong><br>Explora encolar, desencolar y las cuatro operaciones de deque. Observa qué punteros permanecen sin cambio.</div>

### Comprueba tu comprensión

**Pregunta:** En la secuencia manual, ¿qué referencias cambian al desencolar A de la cadena A → B → C?

**Responde esta pregunta en notebook.md.**


In [ ]:
from pathlib import Path
import sys
candidatas = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'clase_17', Path.cwd().parent / 'clase_17']
RAIZ_CLASE = next((ruta for ruta in candidatas if (ruta / 'src' / 'visualizaciones_listas.py').exists()), None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('Abre el notebook desde curso-alumnos o clase_17/notebooks.')
sys.path.insert(0, str(RAIZ_CLASE))
from src.visualizaciones_listas import diagnosticar_widgets, mostrar_listas
print(diagnosticar_widgets())
mostrar_listas()


## 8. Complejidad

La lista no hace mágicamente rápidas todas las operaciones. Es eficiente porque guardamos exactamente las referencias que evitan recorridos para el patrón FIFO.

| Operación | Cola ligada | Razón |
| --- | ---: | --- |
| `encolar` | O(1) | acceso directo a `_final` |
| `desencolar` | O(1) | acceso directo a `_frente` |
| `primero` | O(1) | solo consulta `_frente` |
| `len` | O(1) | tamaño almacenado |
| buscar un valor | O(n) | puede requerir toda la cadena |

La memoria total es O(n): cada elemento necesita un nodo y una referencia adicional. Para BFS, cada vértice se encola y desencola a lo sumo una vez; con listas de adyacencia, el recorrido completo cuesta O(V + E).

<div style="border-left:5px solid #315f8c;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong>NOTA — Complejidad por diseño</strong><br>O(1) no proviene del nombre «cola»; proviene de conservar frente, final y tamaño de forma consistente.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Por qué buscar un valor sigue siendo O(n) aunque encolar y desencolar sean O(1)?

**Responde esta pregunta en notebook.md.**


## 9. BFS y cola

BFS explora un grafo por capas. Desde `A`, primero procesa distancia 0 (`A`), luego distancia 1 (`B`, `C`), después distancia 2 y así sucesivamente. La cola preserva exactamente esa prioridad: lo descubierto antes se procesa antes.

```text
capa 0: A
capa 1: B, C
capa 2: D, E
capa 3: F
```

El ciclo principal desencola un vértice, inspecciona sus vecinos y encola los que todavía no fueron descubiertos. En esta práctica es obligatorio usar `ColaLigada`; utilizar `list.pop(0)` ocultaría el vínculo entre política y representación.

```text
encolar origen → mientras haya pendientes:
    desencolar actual
    por cada vecino no descubierto:
        marcar, guardar predecesor, encolar
```

<div style="border-left:5px solid #7b4bb7;background:#f4effa;padding:12px 16px;margin:14px 0"><strong>IMPORTANTE — Prioridad implícita</strong><br>BFS no guarda distancias en la cola; el orden FIFO hace que las capas aparezcan en orden no decreciente.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué relación existe entre el orden FIFO y el procesamiento por capas de BFS?

**Responde esta pregunta en notebook.md.**


## 10. Visitados al encolar

Un vértice debe marcarse como descubierto **cuando se encola**, no cuando se desencola. Supón que `B` y `C` apuntan a `E`. Si `E` no se marca al descubrirse desde `B`, al procesar `C` podría encolarse otra vez. En grafos densos, las duplicaciones crecen rápidamente.

El momento correcto reúne tres acciones:

1. agregar el vecino al conjunto de descubiertos;
2. registrar su primer predecesor;
3. encolarlo una sola vez.

Ese primer predecesor construye un árbol BFS. Como todos los caminos que llegan a la misma capa usan el mismo número de aristas, conservar el primero respeta la ruta mínima según el orden de vecinos.

<div style="border-left:5px solid #b26a00;background:#fff6e6;padding:12px 16px;margin:14px 0"><strong>ADVERTENCIA — Demasiado tarde</strong><br>Marcar al desencolar puede producir varias copias pendientes y predecesores sobrescritos.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué duplicación puede ocurrir si un vértice se marca solo al desencolarse?

**Responde esta pregunta en notebook.md.**


## 11. Predecesores

BFS no necesita guardar caminos completos dentro de la cola. Mantiene un mapa compacto: `predecesores[vecino] = actual` al descubrir cada vecino. El origen tiene predecesor `None`; un nodo inalcanzable también conserva `None`, pero se distingue porque no es el origen y nunca fue descubierto.

Para el camino `A → B → D → F`, la tabla relevante es:

| nodo | predecesor |
| --- | --- |
| A | `None` |
| B | A |
| D | B |
| F | D |

Desde `F` seguimos hacia atrás hasta `A` y luego invertimos. Esta separación permite reutilizar la reconstrucción en BFS, 0-1 BFS y Dijkstra: el algoritmo calcula costos y predecesores; otra función interpreta el mapa.

<div style="border-left:5px solid #315f8c;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong>NOTA — Responsabilidad única</strong><br>Calcular predecesores y reconstruir un camino son tareas relacionadas, pero distintas y probables de fallar por razones diferentes.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué información mínima permite reconstruir un camino sin guardar rutas completas durante el recorrido?

**Responde esta pregunta en notebook.md.**


## 12. Reconstrucción del camino

Una reconstrucción robusta trata cuatro contratos. Si origen o destino no aparecen como claves, hay un error de uso y se lanza `KeyError`. Si son iguales, el camino es `[origen]`. Si la cadena termina en `None` antes de llegar al origen, el destino es inalcanzable y se devuelve `[]`. Si se repite un nodo, la tabla está corrupta y se lanza `ValueError` por ciclo.

```text
actual = destino
mientras actual no sea None:
    si ya apareció: error de ciclo
    agregar actual
    si actual == origen: invertir y devolver
    avanzar a predecesores[actual]
devolver []
```

También debe detectarse una referencia a una clave ausente: `{A: None, B: X}` no es simplemente «inalcanzable»; es una tabla rota.

<div style="border-left:5px solid #b42318;background:#fff0ef;padding:12px 16px;margin:14px 0"><strong>PRECAUCIÓN — None es ambiguo</strong><br>Puede señalar el origen o un nodo inalcanzable. El contexto decide cuál significado corresponde.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Cómo distingues un destino inalcanzable de una tabla de predecesores corrupta?

**Responde esta pregunta en notebook.md.**


## 13. Práctica guiada de BFS

Usa el grafo conductor y respeta el orden de vecinos:

```python
{"A": ["B", "C"], "B": ["A", "D", "E"],
 "C": ["A", "E"], "D": ["B", "F"],
 "E": ["B", "C", "F"], "F": ["D", "E"]}
```

| paso | sale | cola después | nuevos predecesores |
| ---: | --- | --- | --- |
| 0 | — | A | A: `None` |
| 1 | A | B, C | B: A; C: A |
| 2 | B | C, D, E | D: B; E: B |
| 3 | C | ? | ? |
| 4 | D | ? | ? |

Completa la traza y luego usa el visualizador. Mantén fija la posición del grafo y observa tres variables: frente de la cola, conjunto descubierto y predecesores. Con este orden, el camino `A` a `F` será `A → B → D → F`; otro orden de vecinos podría producir otro camino igualmente corto.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué puede cambiar el camino concreto sin cambiar su longitud mínima cuando cambia el orden de vecinos?

**Responde esta pregunta en notebook.md.**


## 14. Lista doblemente ligada

Una lista doble agrega en cada nodo la referencia `anterior`:

```text
None ← [A] ⇄ [B] ⇄ [C] → None
```

El beneficio es simétrico: con referencias externas a inicio y final podemos retirar o agregar en ambos extremos en O(1). El costo es memoria adicional y más actualizaciones por operación. Al enlazar dos nodos deben concordar ambas direcciones: si `A.siguiente is B`, entonces `B.anterior is A`.

Retirar un nodo exige reparar el nuevo extremo y limpiar **ambas** referencias del retirado. En el nuevo inicio, `anterior` debe ser `None`; en el nuevo final, `siguiente` debe ser `None`.

<div style="border-left:5px solid #b26a00;background:#fff6e6;padding:12px 16px;margin:14px 0"><strong>ADVERTENCIA — Dos direcciones</strong><br>Actualizar un solo enlace puede producir una estructura que parece correcta de izquierda a derecha y está rota al recorrerla de regreso.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué nueva capacidad obtenemos con anterior y qué obligación de consistencia aparece?

**Responde esta pregunta en notebook.md.**


## 15. Invariantes de lista doble

Los invariantes de extremos se amplían:

| Estado | Condiciones |
| --- | --- |
| vacía | inicio y final son `None`; tamaño 0 |
| un nodo | inicio es final; `anterior` y `siguiente` son `None` |
| varios | inicio.anterior es `None`; final.siguiente es `None`; enlaces internos concuerdan |

Una verificación útil recorre de inicio a final contando nodos y comprobando que cada `actual.anterior` sea el nodo visitado justo antes. Luego puede recorrerse de final a inicio. Ambas cuentas deben coincidir con `_tamano`.

El estado de un solo elemento merece pruebas desde ambos extremos: agregar por inicio y quitar por final; agregar por final y quitar por inicio. Esas combinaciones revelan implementaciones que tratan los extremos como estructuras independientes.

<div style="border-left:5px solid #7b4bb7;background:#f4effa;padding:12px 16px;margin:14px 0"><strong>IMPORTANTE — Una cadena, dos recorridos</strong><br>Inicio y final deben describir el mismo conjunto de nodos, no dos historias parcialmente sincronizadas.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Cómo comprobarías automáticamente que los enlaces anterior y siguiente son consistentes?

**Responde esta pregunta en notebook.md.**


## 16. Deque ligada

Una **deque** —double-ended queue— permite agregar y retirar por ambos extremos. No impone por sí misma FIFO ni LIFO: el algoritmo decide cómo usarla. En esta clase ofrece cuatro operaciones O(1): `agregar_inicio`, `agregar_final`, `quitar_inicio` y `quitar_final`, más consultas de extremos.

```text
agregar/quitar                       agregar/quitar
    inicio ← [A] ⇄ [B] ⇄ [C] → final
```

Para 0-1 BFS siempre se retira del inicio. Una mejora por arista de peso 0 se agrega al inicio; una mejora por peso 1 se agrega al final. Así obtenemos dos niveles de prioridad sin pagar el costo de un heap.

En producción Python usaríamos normalmente `collections.deque`, implementada y probada para este patrón. Aquí construimos la versión ligada para observar los invariantes que una biblioteca nos oculta y conectar representación con algoritmo.

<div style="border-left:5px solid #315f8c;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong>NOTA — TDA e implementación</strong><br>«Deque» describe operaciones. Una lista doble es una forma de implementarlas; no son sinónimos.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Por qué una deque no determina por sí sola si el comportamiento será FIFO o LIFO?

**Responde esta pregunta en notebook.md.**


## 17. Operaciones manuales de deque

Traza cada cambio en ambas direcciones:

| paso | operación | inicio | cadena | final | tamaño | devuelve |
| ---: | --- | --- | --- | --- | ---: | --- |
| 0 | crear | `None` | vacía | `None` | 0 | — |
| 1 | `agregar_inicio("B")` | B | B | B | 1 | — |
| 2 | `agregar_inicio("A")` | A | A ⇄ B | B | 2 | — |
| 3 | `agregar_final("C")` | ? | ? | ? | ? | — |
| 4 | `agregar_final("D")` | ? | ? | ? | ? | — |
| 5 | `quitar_inicio()` | ? | ? | ? | ? | ? |
| 6 | `quitar_final()` | ? | ? | ? | ? | ? |

Continúa hasta vaciar y reutilizar. En cada extracción anota los enlaces del nodo retirado: ambos deben terminar en `None`. Luego explora las cuatro operaciones en la visualización y distingue enlaces transitorios de invariantes restaurados.

### Comprueba tu comprensión

**Pregunta:** Después de quitar el inicio A de A ⇄ B ⇄ C ⇄ D, ¿qué cuatro hechos deben comprobarse?

**Responde esta pregunta en notebook.md.**


## 18. Qué problema resuelve 0-1 BFS

BFS minimiza número de aristas porque supone que todas cuestan lo mismo. Pero considera dos rutas: una arista de costo 1 frente a tres aristas de costo 0. BFS elegiría la primera por tener menos saltos; el costo correcto favorece la segunda.

0-1 BFS resuelve caminos mínimos cuando **cada peso es exactamente 0 o 1**. Al relajar `actual → vecino`, calcula `candidato = distancia[actual] + peso`. Si mejora, actualiza distancia y predecesor. Peso 0 significa «este candidato conserva el costo actual» y debe revisarse pronto; peso 1 significa «queda detrás de los candidatos del costo actual».

<div style="border-left:5px solid #7b4bb7;background:#f4effa;padding:12px 16px;margin:14px 0"><strong>IMPORTANTE — Dominio exacto</strong><br>No basta con pesos “pequeños” ni no negativos. La regla inicio/final depende de que solo existan 0 y 1.</div>

El algoritmo no usa un conjunto `visitados` definitivo como BFS: una distancia previamente descubierta puede mejorar y su predecesor debe actualizarse.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué BFS común puede fallar cuando algunas aristas cuestan 0?

**Responde esta pregunta en notebook.md.**


## 19. Deque como estructura de prioridad

La deque funciona como una cola de prioridad especializada con solo dos incrementos posibles. Siempre retiramos del inicio:

- mejora con peso 0 → `agregar_inicio(vecino)`;
- mejora con peso 1 → `agregar_final(vecino)`.

```text
inicio [mismo costo] [mismo costo] ... [costo + 1] final
```

No se agrega un vecino si el candidato empata o empeora; solo una mejora estricta cambia distancia y predecesor. Un nodo puede aparecer históricamente más de una vez si fue agregado con una distancia y luego vuelve a mejorar. Al retirarlo, la implementación usa su distancia **actual**; la repetición puede costar trabajo, pero no invalida el resultado.

<div style="border-left:5px solid #b26a00;background:#fff6e6;padding:12px 16px;margin:14px 0"><strong>ADVERTENCIA — No es FIFO puro</strong><br>Las aristas gratuitas adelantan candidatos; las de costo uno respetan la espera al final.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué información del peso decide el extremo de inserción y por qué?

**Responde esta pregunta en notebook.md.**


## 20. Ejecución manual de 0-1 BFS

Usa el grafo conductor:

```python
{"A": [("B", 0), ("C", 1)], "B": [("D", 0), ("E", 1)],
 "C": [("D", 0)], "D": [("F", 1)],
 "E": [("F", 0)], "F": []}
```

| paso | sale | arista | decisión | deque inicio→final | cambio de distancia |
| ---: | --- | --- | --- | --- | --- |
| 0 | — | — | agregar origen | A | d(A)=0 |
| 1 | A | A→B (0) | inicio | B | d(B)=0 |
| 2 | A | A→C (1) | final | B,C | d(C)=1 |
| 3 | B | B→D (0) | ? | ? | ? |
| 4 | B | B→E (1) | ? | ? | ? |

Completa las distancias y predecesores. Después explora dos secuencias del visualizador: el ejemplo conductor y `A→X(1), A→B(0), B→C(0), C→X(0)`, donde `X` mejora de 1 a 0 y cambia de predecesor.

### Comprueba tu comprensión

**Pregunta:** Cuando X mejora de distancia 1 a 0, ¿qué valores cambian y dónde se agrega X?

**Responde esta pregunta en notebook.md.**


In [ ]:
from src.visualizaciones_bfs import mostrar_bfs
mostrar_bfs()


## 21. Implementación

La entrega separa estructuras, normalización y algoritmos. Conserva exactamente las firmas de `estructuras_bfs_base.py` y copia ese archivo como `implementacion.py` en tu carpeta de entrega. No cambies nombres ni importaciones.

Orden recomendado:

1. `NodoSimple` y `ColaLigada` con pruebas de invariantes;
2. `bfs_predecesores`, `reconstruir_camino`, `bfs_camino`;
3. `NodoDoble` y `DequeLigada` con pruebas bidireccionales;
4. normalización 0-1, `cero_uno_bfs`, `camino_cero_uno`;
5. casos límite, pruebas propias y documentación.

El notebook no contiene la solución completa. Los pseudocódigos explican responsabilidades; tú debes traducirlas respetando type hints, docstrings, excepciones y no mutación.

<div style="border-left:5px solid #b42318;background:#fff0ef;padding:12px 16px;margin:14px 0"><strong>PRECAUCIÓN — Restricción central</strong><br>No uses <code>collections.deque</code> ni heap en la entrega. BFS debe usar tu <code>ColaLigada</code> y 0-1 BFS tu <code>DequeLigada</code>.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué ventaja tiene implementar y probar cada estructura antes de integrarla al algoritmo?

**Responde esta pregunta en notebook.md.**


## 22. Casos límite

La normalización crea una copia independiente del grafo y agrega como claves los vecinos implícitos. Valida nodos `str`, adyacencias secuenciales y aristas 0-1 con forma `(vecino, peso)`. Un booleano es subtipo de `int` en Python, pero aquí debe rechazarse explícitamente.

| Situación | Contrato |
| --- | --- |
| grafo, nodo, vecino o peso con tipo incorrecto | `TypeError` |
| arista con longitud distinta de 2 o entero fuera de 0/1 | `ValueError` |
| origen o destino no pertenece al grafo normalizado | `KeyError` |
| origen = destino | `[origen]` o `(0, [origen])` |
| destino inalcanzable | `[]` o `(inf, [])` |

No añadas claves a la entrada ni reutilices sus listas. Copiar defensivamente garantiza que una llamada no cambie el dato del usuario.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué True debe producir TypeError aunque isinstance(True, int) sea verdadero en Python?

**Responde esta pregunta en notebook.md.**


## 23. Pruebas

Una prueba útil nombra la regla que protege, construye una entrada pequeña, fija el resultado y explica por qué el fallo sería peligroso. Debes escribir al menos ocho pruebas propias: FIFO; reutilización de cola; combinación y reutilización de deque; BFS con ciclo e inalcanzable; ruta 0-1 con más aristas y menor costo; peso inválido.

```python
def test_cola_se_reutiliza_despues_de_vaciarse():
    # Regla: vaciar restaura ambos extremos.
    cola = ColaLigada()
    cola.encolar("A")
    assert cola.desencolar() == "A"
    cola.encolar("B")
    assert cola.desencolar() == "B"
    assert cola.esta_vacia()
```

Agrega casos para vecino implícito, no mutación, mejora posterior y referencias residuales. Ejecuta pruebas públicas y propias con `evaluar.py`, conserva la salida completa y explica la prueba que más información te dio.

<div style="border-left:5px solid #2f7d5b;background:#edf8f2;padding:12px 16px;margin:14px 0"><strong>CONSEJO — Falsar, no decorar</strong><br>Una buena prueba intenta romper un contrato específico; no repite ocho veces el caso feliz.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué defecto concreto detecta una prueba que vacía y vuelve a llenar la misma estructura?

**Responde esta pregunta en notebook.md.**


## 24. Comparación BFS, 0-1 BFS y Dijkstra

Los tres algoritmos comparten relajación o descubrimiento y predecesores; cambia la estructura que selecciona el próximo candidato.

| Pesos | Algoritmo | Estructura | Operación dominante | Costo típico |
| --- | --- | --- | --- | ---: |
| todos iguales / sin peso | BFS | cola | orden de llegada por capas | O(V+E) |
| exactamente 0 o 1 | 0-1 BFS | deque | adelantar costo 0, posponer costo 1 | O(V+E) |
| no negativos generales | Dijkstra | min-heap | extraer menor distancia candidata | O((V+E) log V) |

Ninguno es universalmente superior. BFS es más simple cuando aplica; 0-1 BFS explota un dominio especial; Dijkstra cubre más pesos con una estructura más costosa. Para pesos `0, 1, 2`, la regla binaria de extremos ya no representa tres incrementos, por lo que esta implementación de 0-1 BFS no aplica directamente.

<div style="border-left:5px solid #7b4bb7;background:#f4effa;padding:12px 16px;margin:14px 0"><strong>IMPORTANTE — Primero el contrato</strong><br>Inspecciona los pesos antes de elegir el algoritmo; después elige la estructura que ejecuta esa prioridad.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación dominante conduce respectivamente a cola, deque o heap?

**Responde esta pregunta en notebook.md.**


## 25. Cierre hacia Union-Find y Kruskal

La ruta completa fue: describir la prioridad, elegir el TDA, implementar la representación, proteger invariantes y usarla dentro de un algoritmo.

| Pregunta | Respuesta de la clase |
| --- | --- |
| ¿Todos los pendientes valen igual? | cola ligada + BFS |
| ¿Los incrementos son 0 o 1? | deque ligada + 0-1 BFS |
| ¿Los pesos no negativos son generales? | heap + Dijkstra |
| ¿Qué se reutiliza? | normalización, predecesores, reconstrucción y pruebas por contrato |

En la siguiente clase cambiará el objetivo. Ya no buscaremos un camino desde un origen, sino seleccionar conexiones baratas para unir componentes sin crear ciclos. Esa necesidad motivará Union-Find y Kruskal; todavía no los implementamos aquí.

<div style="border-left:5px solid #315f8c;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong>CIERRE — La estructura sigue a la decisión</strong><br>Cola, deque y heap no son alternativas intercambiables: materializan políticas de selección diferentes.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué nueva pregunta aparece si queremos seleccionar aristas baratas sin formar ciclos?

**Responde esta pregunta en notebook.md.**
